<a href="https://colab.research.google.com/github/anushka1511/college-enquiry/blob/main/college_enquiry_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install -U langchain langchain-community langchain-google-genai langchain-huggingface faiss-cpu sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 751.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:

import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")


Enter your Gemini API key: ··········


In [3]:

college_faqs = [
    {
        "category": "admissions",
        "id": "ADM01",
        "question": "What is the admission process?",
        "answer": "Students must complete the application form, submit the required documents, and follow the official admission and selection process."
    },
    {
        "category": "admissions",
        "id": "ADM02",
        "question": "What documents are required for admission?",
        "answer": "Typical documents include qualifying examination marksheets, identity proof, photographs, and other documents specified by the college."
    },
    {
        "category": "admissions",
        "id": "ADM03",
        "question": "Where can students find fee information?",
        "answer": "Students should consult the official fee structure published by the college or contact the admissions office."
    },
    {
        "category": "admissions",
        "id": "ADM04",
        "question": "How can I apply for a scholarship?",
        "answer": "Students should check the college scholarship guidelines for eligibility, required documents, deadlines, and application procedures."
    },
    {
        "category": "admissions",
        "id": "ADM05",
        "question": "How do I contact the admissions office?",
        "answer": "Students should use the contact details listed on the college's official website."
    },
    {
        "category": "academics",
        "id": "ACA01",
        "question": "What is the attendance requirement?",
        "answer": "The applicable attendance requirement depends on the institution's official academic regulations. Students should verify the current rule in the academic handbook."
    },
    {
        "category": "academics",
        "id": "ACA02",
        "question": "How are semester examinations conducted?",
        "answer": "Semester examinations follow the institution's published academic calendar and examination regulations."
    },
    {
        "category": "academics",
        "id": "ACA03",
        "question": "How can students check their grades?",
        "answer": "Students can check grades through the official student portal once results have been released."
    },
    {
        "category": "academics",
        "id": "ACA04",
        "question": "What should I do if I miss an examination?",
        "answer": "Students should promptly contact the academic office and follow the official rules for absence, documentation, and any permitted examination arrangements."
    },
    {
        "category": "academics",
        "id": "ACA05",
        "question": "Where can I find the academic calendar?",
        "answer": "Students should consult the official academic calendar or student portal for semester dates, holidays, and examination schedules."
    }
]

print("Total FAQ entries:", len(college_faqs))


Total FAQ entries: 10


In [4]:

from langchain_core.documents import Document

documents = []

for faq in college_faqs:
    text = (
        f"Question: {faq['question']}\n"
        f"Answer: {faq['answer']}"
    )

    documents.append(
        Document(
            page_content=text,
            metadata={
                "category": faq["category"],
                "source": faq["id"]
            }
        )
    )

print(documents[0])


page_content='Question: What is the admission process?
Answer: Students must complete the application form, submit the required documents, and follow the official admission and selection process.' metadata={'category': 'admissions', 'source': 'ADM01'}


In [5]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Separate searchable knowledge bases for each specialist
admissions_docs = [
    d for d in documents
    if d.metadata["category"] == "admissions"
]

academics_docs = [
    d for d in documents
    if d.metadata["category"] == "academics"
]

admissions_db = FAISS.from_documents(
    admissions_docs, embedding_model
)

academics_db = FAISS.from_documents(
    academics_docs, embedding_model
)

admissions_retriever = admissions_db.as_retriever(
    search_kwargs={"k": 3}
)

academics_retriever = academics_db.as_retriever(
    search_kwargs={"k": 3}
)

print("Both specialist knowledge bases are ready.")


/tmp/ipykernel_2441/4154702800.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both specialist knowledge bases are ready.


In [7]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini model initialized.")


Gemini model initialized.


In [8]:

router_prompt = ChatPromptTemplate.from_template("""
You are the routing agent of a college enquiry system.

Classify the student's question into exactly one category:

ADMISSIONS:
Applications, eligibility, documents, fees, scholarships,
admission procedures, admissions office.

ACADEMICS:
Attendance, examinations, grades, academic calendar,
courses, academic regulations.

OTHER:
Questions unrelated to these categories, or questions
that cannot be classified confidently.

Return exactly one word:
ADMISSIONS, ACADEMICS, or OTHER.

Student question: {question}
""")

router_chain = (
    router_prompt
    | llm
    | StrOutputParser()
)

def route_question(question):
    result = router_chain.invoke({"question": question})
    result = result.strip().upper()

    if "ADMISSIONS" in result:
        return "admissions"

    if "ACADEMICS" in result:
        return "academics"

    return "other"


In [9]:

def format_docs(docs):
    if not docs:
        return "No relevant documents found."

    return "\n\n".join(
        f"[Source: {doc.metadata['source']}]\n{doc.page_content}"
        for doc in docs
    )


answer_prompt = ChatPromptTemplate.from_template("""
You are the {agent_name} of a college enquiry chatbot.

Answer the student's question using ONLY the retrieved context below.

Rules:
1. Do not invent college policies, dates, fees, or requirements.
2. If the context does not contain the answer, clearly say so.
3. Give a concise, student-friendly answer.
4. Mention the relevant source ID, such as [ADM01] or [ACA01],
   when using information from a retrieved document.
5. If a question needs current or official information that
   is not present in the context, direct the student to verify
   it with the appropriate college office.

Retrieved context:
{context}

Student question:
{question}

Answer:
""")


def build_rag_agent(retriever, agent_name):
    prompt = answer_prompt.partial(agent_name=agent_name)

    chain = (
        RunnableParallel({
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        })
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain


admissions_agent = build_rag_agent(
    admissions_retriever,
    "Admissions Specialist"
)

academics_agent = build_rag_agent(
    academics_retriever,
    "Academic Support Specialist"
)

print("Both specialist RAG agents are ready.")


Both specialist RAG agents are ready.


In [10]:

def college_chatbot(question):
    category = route_question(question)

    print(f"Selected agent: {category.upper()}")

    if category == "admissions":
        answer = admissions_agent.invoke(question)

    elif category == "academics":
        answer = academics_agent.invoke(question)

    else:
        answer = (
            "I can help with admissions and academic enquiries. "
            "For other questions, please contact the relevant "
            "college department or provide a question related to "
            "admissions, fees, attendance, examinations, or grades."
        )

    return answer


print("Multi-Agent College Enquiry Chatbot is ready!")


Multi-Agent College Enquiry Chatbot is ready!


In [11]:

test_questions = [
    "What documents are required for admission?",
    "How can I check my semester grades?",
    "How do I apply for a scholarship?",
    "What is the attendance requirement?",
    "What is the process for missing an examination?",
    "What is the best restaurant near my college?"
]

for question in test_questions:
    print("\n" + "=" * 65)
    print("QUESTION:", question)
    print("-" * 65)

    try:
        answer = college_chatbot(question)
        print("ANSWER:", answer)
    except Exception as e:
        print("Error:", str(e))



QUESTION: What documents are required for admission?
-----------------------------------------------------------------
Selected agent: ADMISSIONS
ANSWER: Typical documents required for admission include qualifying examination marksheets, identity proof, photographs, and other documents specified by the college [ADM02].

QUESTION: How can I check my semester grades?
-----------------------------------------------------------------
Selected agent: ACADEMICS
ANSWER: You can check your grades through the official student portal once results have been released [ACA03].

QUESTION: How do I apply for a scholarship?
-----------------------------------------------------------------
Selected agent: ADMISSIONS
ANSWER: To apply for a scholarship, students should check the college scholarship guidelines for eligibility, required documents, deadlines, and application procedures [ADM04].

QUESTION: What is the attendance requirement?
-----------------------------------------------------------------


In [12]:

print("COLLEGE ENQUIRY CHATBOT")
print("Type 'exit' to stop.\n")

while True:
    question = input("You: ").strip()

    if question.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye! Have a great day.")
        break

    if not question:
        print("Bot: Please enter a question.")
        continue

    try:
        answer = college_chatbot(question)
        print("\nBot:", answer, "\n")
    except Exception as e:
        print("\nBot: Something went wrong. Please try again.")
        print("Technical details:", str(e), "\n")


COLLEGE ENQUIRY CHATBOT
Type 'exit' to stop.

You: exit
Bot: Goodbye! Have a great day.


In [13]:

routing_tests = [
    ("What are the admission requirements?", "admissions"),
    ("Where can I find the fee structure?", "admissions"),
    ("How do I apply for a scholarship?", "admissions"),
    ("How do I check my grades?", "academics"),
    ("When are semester examinations conducted?", "academics"),
    ("Where can I find the academic calendar?", "academics"),
]

correct = 0
results = []

for question, expected in routing_tests:
    predicted = route_question(question)
    passed = predicted == expected
    correct += int(passed)

    results.append({
        "Question": question,
        "Expected": expected,
        "Predicted": predicted,
        "Correct": passed
    })

accuracy = correct / len(routing_tests) * 100

print(f"Routing accuracy: {accuracy:.1f}%")
for result in results:
    print(result)


Routing accuracy: 100.0%
{'Question': 'What are the admission requirements?', 'Expected': 'admissions', 'Predicted': 'admissions', 'Correct': True}
{'Question': 'Where can I find the fee structure?', 'Expected': 'admissions', 'Predicted': 'admissions', 'Correct': True}
{'Question': 'How do I apply for a scholarship?', 'Expected': 'admissions', 'Predicted': 'admissions', 'Correct': True}
{'Question': 'How do I check my grades?', 'Expected': 'academics', 'Predicted': 'academics', 'Correct': True}
{'Question': 'When are semester examinations conducted?', 'Expected': 'academics', 'Predicted': 'academics', 'Correct': True}
{'Question': 'Where can I find the academic calendar?', 'Expected': 'academics', 'Predicted': 'academics', 'Correct': True}
